# Feature Engineering: Indirect Prompt Injection Detection

Este notebook construye las características para detectar ataques de prompt injection usando clasificadores ML clásicos.

**Objetivo:** Extraer características lingüísticas y embeddings semánticos del dataset analizado en el EDA.

**Estructura:**
1. Load Dataset & EDA Summary
2. Linguistic / Handcrafted Features
3. Semantic Embedding Features
4. Feature Combination & Final Feature Matrix
5. Feature Importance Analysis
6. Save Artifacts

In [3]:
import re
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
## SECTION 1 — Load Dataset & EDA Summary

Cargamos el dataset de indirect prompt injection y resumimos los hallazgos clave del EDA.

In [4]:
def resolve_data_path() -> Path:
    candidates = [
        Path("../data/indirect_prompt_injection_bipia_gpt_train.csv"),
        Path("data/indirect_prompt_injection_bipia_gpt_train.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Dataset CSV not found. Expected one of: "
        + ", ".join(str(p) for p in candidates)
    )

DATA_PATH = resolve_data_path()
df = pd.read_csv(DATA_PATH)

print(f"Dataset path: {DATA_PATH}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset path: ../data/indirect_prompt_injection_bipia_gpt_train.csv
Shape: (70000, 4)
Columns: ['context', 'user_intent', 'label', 'source']


In [5]:
# Verificar distribución de clases
print("Class distribution:")
print(df["label"].value_counts().sort_index())
print(f"\nTotal samples: {len(df)}")
print(f"Benign (0): {(df['label'] == 0).sum()} ({(df['label'] == 0).mean()*100:.1f}%)")
print(f"Malicious (1): {(df['label'] == 1).sum()} ({(df['label'] == 1).mean()*100:.1f}%)")

Class distribution:
label
0    35000
1    35000
Name: count, dtype: int64

Total samples: 70000
Benign (0): 35000 (50.0%)
Malicious (1): 35000 (50.0%)


In [6]:
# Verificar valores nulos
print("Null values per column:")
print(df.isna().sum())
assert df.isna().sum().sum() == 0, "Dataset contains null values!"

Null values per column:
context        0
user_intent    0
label          0
source         0
dtype: int64


### Conclusiones del EDA

Del análisis exploratorio previo se identificaron los siguientes hallazgos relevantes para feature engineering:

1. **Balance de clases:** Dataset perfectamente balanceado (35,000 benignos / 35,000 maliciosos)

2. **Confounding variable:** `label` está alineado 1:1 con `source` (BIPIA → malicious, GPT-4o-mini → benign). Esto implica que el modelo podría aprender patrones de estilo/origen en lugar de semántica de inyección.

3. **Longitudes discriminativas:**
   - `context`: mediana 961 chars, p95 = 3,305 chars
   - `user_intent`: mediana 56 chars, p95 = 343 chars
   - Los ejemplos maliciosos tienden a tener contextos con instrucciones embebidas

4. **Tipos de contenido detectados:**
   - Tablas Markdown: ~28.9%
   - Code fences: ~19.7%
   - Formato email: ~11.3%
   - URLs: ~8.9% (más frecuente en maliciosos: 14.8% vs 3%)
   - SQL keywords: ~7.9% (más en maliciosos: 10.1% vs 5.6%)

5. **Patrones de inyección:**
   - Frases directas como "ignore previous" no aparecen con alta frecuencia
   - "do not" aparece más en maliciosos (10.7% vs 3.9%)
   - Las inyecciones son sutiles y embebidas en contenido externo

**Implicación para features:** Necesitamos características que capturen tanto patrones léxicos explícitos como estructura semántica subyacente.

In [7]:
# Preparar columna de texto para feature extraction
# Usamos 'context' ya que es donde se embeben las instrucciones maliciosas
df["text"] = df["context"].astype(str)
y = df["label"].values

print(f"Text column created from 'context'")
print(f"Label array shape: {y.shape}")

Text column created from 'context'
Label array shape: (70000,)


---
## SECTION 2 — Linguistic / Handcrafted Features

Extraemos 10 características lingüísticas diseñadas específicamente para detectar prompt injection.
Cada feature está justificada en términos de su relevancia para la detección.

### Feature 1: `char_count` — Total character count

**Justificación:** Los prompts de inyección indirecta frecuentemente contienen instrucciones adicionales embebidas en el contexto, lo que puede resultar en textos más largos. Según el EDA, hay variación significativa en longitudes entre clases.

In [8]:
df["char_count"] = df["text"].str.len()
print(f"char_count - min: {df['char_count'].min()}, max: {df['char_count'].max()}, mean: {df['char_count'].mean():.2f}")

char_count - min: 31, max: 26406, mean: 1289.14


### Feature 2: `word_count` — Total word count

**Justificación:** Complementa char_count con una medida a nivel de palabra. Las instrucciones de inyección requieren vocabulario específico que puede aumentar el conteo de palabras.

In [9]:
df["word_count"] = df["text"].str.split().str.len()
print(f"word_count - min: {df['word_count'].min()}, max: {df['word_count'].max()}, mean: {df['word_count'].mean():.2f}")

word_count - min: 1, max: 6462, mean: 196.79


### Feature 3: `avg_word_length` — Average word length

**Justificación:** Textos técnicos o con comandos específicos pueden tener palabras más largas (e.g., "instructions", "override", "disregard"). Esta métrica captura complejidad léxica.

In [10]:
df["avg_word_length"] = df["char_count"] / df["word_count"].replace(0, 1)
print(f"avg_word_length - min: {df['avg_word_length'].min():.2f}, max: {df['avg_word_length'].max():.2f}, mean: {df['avg_word_length'].mean():.2f}")

avg_word_length - min: 3.38, max: 1772.00, mean: 7.08


### Feature 4: `sentence_count` — Number of sentences

**Justificación:** Las inyecciones de prompt frecuentemente incluyen múltiples oraciones imperativas o instrucciones. Un conteo alto de oraciones puede indicar instrucciones embebidas.

In [11]:
def count_sentences(text: str) -> int:
    """Count sentences by splitting on . ! ?"""
    sentences = re.split(r'[.!?]+', str(text))
    # Filter out empty strings
    return len([s for s in sentences if s.strip()])

df["sentence_count"] = df["text"].apply(count_sentences)
print(f"sentence_count - min: {df['sentence_count'].min()}, max: {df['sentence_count'].max()}, mean: {df['sentence_count'].mean():.2f}")

sentence_count - min: 1, max: 154, mean: 12.75


### Feature 5: `uppercase_ratio` — Ratio of uppercase chars to total chars

**Justificación:** Los atacantes a veces usan MAYÚSCULAS para enfatizar comandos o llamar la atención del modelo (e.g., "IGNORE", "DO NOT", "MUST"). Un ratio alto puede ser indicativo.

In [12]:
def uppercase_ratio(text: str) -> float:
    """Calculate ratio of uppercase characters to total characters."""
    text = str(text)
    if len(text) == 0:
        return 0.0
    uppercase_count = sum(1 for c in text if c.isupper())
    return uppercase_count / len(text)

df["uppercase_ratio"] = df["text"].apply(uppercase_ratio)
print(f"uppercase_ratio - min: {df['uppercase_ratio'].min():.4f}, max: {df['uppercase_ratio'].max():.4f}, mean: {df['uppercase_ratio'].mean():.4f}")

uppercase_ratio - min: 0.0000, max: 0.8530, mean: 0.0459


### Feature 6: `special_char_ratio` — Ratio of special characters to total chars

**Justificación:** Caracteres especiales como `|`, `>`, `$`, `{`, `}` son comunes en código, delimitadores de inyección, y markdown tables. El EDA mostró alta presencia de code fences y tablas.

In [13]:
SPECIAL_CHARS = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')

def special_char_ratio(text: str) -> float:
    """Calculate ratio of special characters to total characters."""
    text = str(text)
    if len(text) == 0:
        return 0.0
    special_count = sum(1 for c in text if c in SPECIAL_CHARS)
    return special_count / len(text)

df["special_char_ratio"] = df["text"].apply(special_char_ratio)
print(f"special_char_ratio - min: {df['special_char_ratio'].min():.4f}, max: {df['special_char_ratio'].max():.4f}, mean: {df['special_char_ratio'].mean():.4f}")

special_char_ratio - min: 0.0000, max: 0.4326, mean: 0.0623


### Feature 7: `injection_keyword_count` — Count of known injection trigger phrases

**Justificación:** Esta es la feature más directamente relacionada con prompt injection. Basado en literatura (Ayub & Majumdar 2024; Rahman et al. 2024) y patrones conocidos de jailbreaking. Incluye frases que intentan anular instrucciones del sistema.

In [14]:
INJECTION_KEYWORDS = [
    "ignore previous",
    "disregard",
    "forget instructions",
    "you are now",
    "pretend you are",
    "act as",
    "jailbreak",
    "DAN",
    "bypass",
    "override",
    "system prompt",
    "new instructions",
    "your real task",
    # Additional keywords based on EDA findings
    "do not",
    "don't",
    "must",
    "always",
    "never",
    "developer",
    "follow these instructions",
    "follow my instructions",
]

def count_injection_keywords(text: str) -> int:
    """Count occurrences of known injection trigger phrases."""
    text_lower = str(text).lower()
    count = 0
    for keyword in INJECTION_KEYWORDS:
        count += len(re.findall(re.escape(keyword.lower()), text_lower))
    return count

df["injection_keyword_count"] = df["text"].apply(count_injection_keywords)
print(f"injection_keyword_count - min: {df['injection_keyword_count'].min()}, max: {df['injection_keyword_count'].max()}, mean: {df['injection_keyword_count'].mean():.2f}")
print(f"Texts with at least 1 keyword: {(df['injection_keyword_count'] > 0).sum()} ({(df['injection_keyword_count'] > 0).mean()*100:.1f}%)")

injection_keyword_count - min: 0, max: 29, mean: 0.39
Texts with at least 1 keyword: 16522 (23.6%)


### Feature 8: `question_mark_count` — Number of ? characters

**Justificación:** Las preguntas son una forma común de intentar extraer información o cambiar el comportamiento del modelo. Un alto número de signos de interrogación puede indicar intentos de manipulación.

In [15]:
df["question_mark_count"] = df["text"].str.count(r'\?')
print(f"question_mark_count - min: {df['question_mark_count'].min()}, max: {df['question_mark_count'].max()}, mean: {df['question_mark_count'].mean():.2f}")

question_mark_count - min: 0, max: 30, mean: 0.20


### Feature 9: `imperative_verb_ratio` — Presence of imperative starters

**Justificación:** Comandos imperativos son la esencia de las inyecciones de prompt. Frases como "tell me", "give me", "show me", "do not" son indicadores de instrucciones directas al modelo.

In [16]:
IMPERATIVE_PATTERNS = [
    r"\btell me\b",
    r"\bgive me\b",
    r"\bshow me\b",
    r"\bdo not\b",
    r"\bdon't\b",
    r"\bmust\b",
    r"\balways\b",
    r"\bnever\b",
    r"\bplease\b",
    r"\bensure\b",
    r"\bmake sure\b",
    r"\byou should\b",
    r"\byou must\b",
    r"\byou will\b",
]

def imperative_verb_ratio(text: str) -> float:
    """Calculate ratio of imperative patterns to word count."""
    text_lower = str(text).lower()
    word_count = len(text_lower.split())
    if word_count == 0:
        return 0.0
    
    imperative_count = 0
    for pattern in IMPERATIVE_PATTERNS:
        imperative_count += len(re.findall(pattern, text_lower))
    
    return imperative_count / word_count

df["imperative_verb_ratio"] = df["text"].apply(imperative_verb_ratio)
print(f"imperative_verb_ratio - min: {df['imperative_verb_ratio'].min():.4f}, max: {df['imperative_verb_ratio'].max():.4f}, mean: {df['imperative_verb_ratio'].mean():.4f}")

imperative_verb_ratio - min: 0.0000, max: 0.0714, mean: 0.0027


### Feature 10: `language_switch_flag` — Binary flag for non-ASCII characters

**Justificación:** El uso de caracteres no-ASCII puede indicar intentos de evasión multilingüe o unicode smuggling. Atacantes pueden usar caracteres de otros idiomas o símbolos unicode para bypassear filtros.

In [17]:
def has_non_ascii(text: str) -> int:
    """Check if text contains non-ASCII characters (potential multilingual evasion)."""
    try:
        str(text).encode('ascii')
        return 0
    except UnicodeEncodeError:
        return 1

df["language_switch_flag"] = df["text"].apply(has_non_ascii)
print(f"language_switch_flag distribution:")
print(df["language_switch_flag"].value_counts())
print(f"\nTexts with non-ASCII: {df['language_switch_flag'].sum()} ({df['language_switch_flag'].mean()*100:.1f}%)")

language_switch_flag distribution:
language_switch_flag
0    44596
1    25404
Name: count, dtype: int64

Texts with non-ASCII: 25404 (36.3%)


### Consolidar DataFrame de Features Lingüísticas

In [18]:
LINGUISTIC_FEATURES = [
    "char_count",
    "word_count",
    "avg_word_length",
    "sentence_count",
    "uppercase_ratio",
    "special_char_ratio",
    "injection_keyword_count",
    "question_mark_count",
    "imperative_verb_ratio",
    "language_switch_flag",
]

df_linguistic = df[LINGUISTIC_FEATURES].copy()

print(f"df_linguistic shape: {df_linguistic.shape}")
print(f"\nFeature statistics:")
df_linguistic.describe()

df_linguistic shape: (70000, 10)

Feature statistics:


,char_count,word_count,avg_word_length,sentence_count,uppercase_ratio,special_char_ratio,injection_keyword_count,question_mark_count,imperative_verb_ratio,language_switch_flag
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,1289.137086,196.794200,7.081777,12.751114,0.045949,0.062345,0.390143,0.200757,0.002738,0.362914
std,959.061421,148.355385,14.762704,10.452247,0.035957,0.043868,0.955287,1.030886,0.006095,0.480844
min,31.000000,1.000000,3.381963,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,587.750000,84.000000,6.017533,6.000000,0.023284,0.028756,0.000000,0.000000,0.000000,0.000000
50%,961.000000,149.000000,6.383648,10.000000,0.034862,0.046016,0.000000,0.000000,0.000000,0.000000
75%,1770.000000,276.000000,7.010667,17.000000,0.054633,0.089572,0.000000,0.000000,0.002222,1.000000
max,26406.000000,6462.000000,1772.000000,154.000000,0.853047,0.432591,29.000000,30.000000,0.071429,1.000000


In [ ]:
# Visualizar distribuciones por clase
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, feature in enumerate(LINGUISTIC_FEATURES):
    ax = axes[i]
    for label in [0, 1]:
        data = df_linguistic.loc[df["label"] == label, feature]
        label_name = "Benign" if label == 0 else "Malicious"
        ax.hist(data, bins=30, alpha=0.5, label=label_name, density=True)
    ax.set_title(feature)
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle("Linguistic Features Distribution by Class", fontsize=14)
plt.tight_layout()
plt.show()

---
## SECTION 3 — Semantic Embedding Features

Generamos representaciones vectoriales densas usando sentence-transformers.
Los embeddings capturan la semántica del texto de forma que textos similares en significado tengan vectores cercanos.

### 3A. Open-source embeddings

Usamos dos modelos de sentence-transformers:
1. **all-MiniLM-L6-v2**: Rápido, 384 dimensiones, bueno para inglés
2. **paraphrase-multilingual-MiniLM-L12-v2**: Multilingüe, relevante dado el flag de language_switch

In [22]:
from sentence_transformers import SentenceTransformer

# Preparar textos
texts = df["text"].tolist()
print(f"Total texts to encode: {len(texts)}")

Total texts to encode: 70000


In [23]:
# Modelo 1: all-MiniLM-L6-v2
print("Loading all-MiniLM-L6-v2...")
model_minilm = SentenceTransformer("all-MiniLM-L6-v2")

start_time = time.time()
embeddings_minilm = model_minilm.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
minilm_time = time.time() - start_time

print(f"\nMiniLM encoding completed!")
print(f"Shape: {embeddings_minilm.shape}")
print(f"Time: {minilm_time:.2f} seconds ({minilm_time/60:.2f} minutes)")

Loading all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9270.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1094/1094 [03:56<00:00,  4.63it/s]



MiniLM encoding completed!
Shape: (70000, 384)
Time: 238.25 seconds (3.97 minutes)


In [24]:
# Modelo 2: paraphrase-multilingual-MiniLM-L12-v2
print("Loading paraphrase-multilingual-MiniLM-L12-v2...")
model_multilingual = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

start_time = time.time()
embeddings_multilingual = model_multilingual.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
multilingual_time = time.time() - start_time

print(f"\nMultilingual encoding completed!")
print(f"Shape: {embeddings_multilingual.shape}")
print(f"Time: {multilingual_time:.2f} seconds ({multilingual_time/60:.2f} minutes)")

Loading paraphrase-multilingual-MiniLM-L12-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7005.82it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1094/1094 [03:25<00:00,  5.32it/s]



Multilingual encoding completed!
Shape: (70000, 384)
Time: 207.97 seconds (3.47 minutes)


In [25]:
# Resumen de tiempos
print("\n=== Embedding Time Summary ===")
print(f"all-MiniLM-L6-v2:                    {minilm_time:.2f}s ({embeddings_minilm.shape[1]} dims)")
print(f"paraphrase-multilingual-MiniLM-L12: {multilingual_time:.2f}s ({embeddings_multilingual.shape[1]} dims)")


=== Embedding Time Summary ===
all-MiniLM-L6-v2:                    238.25s (384 dims)
paraphrase-multilingual-MiniLM-L12: 207.97s (384 dims)


### 3B. Dimensionality Reduction with PCA

Aplicamos PCA para reducir la dimensionalidad de los embeddings a 50 componentes.
Esto reduce ruido, mejora eficiencia y puede mejorar generalización.

In [26]:
# PCA para MiniLM embeddings
pca_minilm = PCA(n_components=50, random_state=RANDOM_STATE)
embeddings_minilm_pca = pca_minilm.fit_transform(embeddings_minilm)

print(f"MiniLM PCA shape: {embeddings_minilm_pca.shape}")
print(f"Explained variance ratio (first 10): {pca_minilm.explained_variance_ratio_[:10]}")
print(f"Total explained variance (50 components): {pca_minilm.explained_variance_ratio_.sum():.4f}")

MiniLM PCA shape: (70000, 50)
Explained variance ratio (first 10): [0.10456143 0.05826368 0.04301826 0.02331412 0.01992841 0.01918129
 0.01723693 0.01655956 0.01506634 0.01470901]
Total explained variance (50 components): 0.6269


In [27]:
# PCA para Multilingual embeddings
pca_multilingual = PCA(n_components=50, random_state=RANDOM_STATE)
embeddings_multilingual_pca = pca_multilingual.fit_transform(embeddings_multilingual)

print(f"Multilingual PCA shape: {embeddings_multilingual_pca.shape}")
print(f"Explained variance ratio (first 10): {pca_multilingual.explained_variance_ratio_[:10]}")
print(f"Total explained variance (50 components): {pca_multilingual.explained_variance_ratio_.sum():.4f}")

Multilingual PCA shape: (70000, 50)
Explained variance ratio (first 10): [0.11533421 0.06868215 0.0626673  0.02526876 0.02045888 0.01848184
 0.01792336 0.01723067 0.01653632 0.0146602 ]
Total explained variance (50 components): 0.7105


In [ ]:
# Scree plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MiniLM scree plot
cumsum_minilm = np.cumsum(pca_minilm.explained_variance_ratio_)
axes[0].bar(range(1, 51), pca_minilm.explained_variance_ratio_, alpha=0.7, label="Individual")
axes[0].plot(range(1, 51), cumsum_minilm, 'r-', linewidth=2, label="Cumulative")
axes[0].axhline(y=0.9, color='g', linestyle='--', label="90% threshold")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("MiniLM Embeddings - PCA Scree Plot")
axes[0].legend()

# Find components for 90% variance
n_components_90_minilm = np.argmax(cumsum_minilm >= 0.9) + 1
print(f"MiniLM: {n_components_90_minilm} components explain 90% variance")

# Multilingual scree plot
cumsum_multi = np.cumsum(pca_multilingual.explained_variance_ratio_)
axes[1].bar(range(1, 51), pca_multilingual.explained_variance_ratio_, alpha=0.7, label="Individual")
axes[1].plot(range(1, 51), cumsum_multi, 'r-', linewidth=2, label="Cumulative")
axes[1].axhline(y=0.9, color='g', linestyle='--', label="90% threshold")
axes[1].set_xlabel("Principal Component")
axes[1].set_ylabel("Explained Variance Ratio")
axes[1].set_title("Multilingual Embeddings - PCA Scree Plot")
axes[1].legend()

n_components_90_multi = np.argmax(cumsum_multi >= 0.9) + 1
print(f"Multilingual: {n_components_90_multi} components explain 90% variance")

plt.tight_layout()
plt.show()

### Nota sobre OpenAI Embeddings

Los embeddings de OpenAI (como `text-embedding-ada-002` o `text-embedding-3-small`) han demostrado excelente rendimiento en tareas de clasificación de texto. Según Ayub & Majumdar (2024), lograron 86.7% de accuracy en detección de prompt injection.

**Ventajas:**
- Alta calidad semántica
- Dimensionalidad configurable (hasta 3072 dims)
- Bien entrenados en texto conversacional

**Desventajas:**
- Requiere API key y tiene costo por token
- Implicaciones de privacidad (datos enviados a servidores externos)
- Latencia de red
- Dependencia de servicio externo

In [29]:
def get_openai_embeddings(texts: list[str]) -> np.ndarray:
    """
    Placeholder for OpenAI embeddings.
    
    To use this function:
    1. Install openai package: pip install openai
    2. Set OPENAI_API_KEY environment variable
    3. Implement the function using openai.Embedding.create()
    
    Example implementation:
    ```python
    import openai
    response = openai.Embedding.create(
        input=texts,
        model="text-embedding-3-small"
    )
    embeddings = [item['embedding'] for item in response['data']]
    return np.array(embeddings)
    ```
    """
    raise NotImplementedError(
        "OpenAI embeddings require API key and have cost implications. "
        "Set OPENAI_API_KEY environment variable and implement this function. "
        "See docstring for example implementation."
    )

---
## SECTION 4 — Feature Combination & Final Feature Matrix

Creamos tres matrices de características candidatas para evaluar en el siguiente notebook.

In [30]:
# Matrix 1: Solo características lingüísticas
X_linguistic = df_linguistic.values.astype(np.float32)

# Matrix 2: Solo embeddings MiniLM (PCA-reduced)
X_embed_minilm = embeddings_minilm_pca.astype(np.float32)

# Matrix 3: Combinación de lingüísticas + embeddings MiniLM
X_combined = np.hstack([X_linguistic, X_embed_minilm]).astype(np.float32)

print("=== Feature Matrix Summary ===")
print(f"X_linguistic shape:  {X_linguistic.shape}")
print(f"X_embed_minilm shape: {X_embed_minilm.shape}")
print(f"X_combined shape:    {X_combined.shape}")

=== Feature Matrix Summary ===
X_linguistic shape:  (70000, 10)
X_embed_minilm shape: (70000, 50)
X_combined shape:    (70000, 60)


### Sanity Check con Logistic Regression

Evaluamos rápidamente cada matriz con validación cruzada para verificar que las features capturan señal discriminativa. **Nota:** Esto NO es el modelo final, solo un sanity check.

In [31]:
def quick_evaluation(X: np.ndarray, y: np.ndarray, name: str) -> tuple[float, float]:
    """Quick sanity check with Logistic Regression and cross-validation."""
    # Scale features for logistic regression
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Logistic regression with cross-validation
    lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1)
    scores = cross_val_score(lr, X_scaled, y, cv=5, scoring='f1')
    
    mean_f1 = scores.mean()
    std_f1 = scores.std()
    
    print(f"{name}:")
    print(f"  Shape: {X.shape}")
    print(f"  F1 Score: {mean_f1:.4f} (+/- {std_f1:.4f})")
    print()
    
    return mean_f1, std_f1

In [32]:
print("=== Sanity Check: Logistic Regression (CV=5, F1 score) ===")
print()

results = {}
results["X_linguistic"] = quick_evaluation(X_linguistic, y, "X_linguistic")
results["X_embed_minilm"] = quick_evaluation(X_embed_minilm, y, "X_embed_minilm")
results["X_combined"] = quick_evaluation(X_combined, y, "X_combined")

=== Sanity Check: Logistic Regression (CV=5, F1 score) ===

X_linguistic:
  Shape: (70000, 10)
  F1 Score: 0.6618 (+/- 0.0072)

X_embed_minilm:
  Shape: (70000, 50)
  F1 Score: 0.6736 (+/- 0.0060)

X_combined:
  Shape: (70000, 60)
  F1 Score: 0.7434 (+/- 0.0045)



In [33]:
# Tabla resumen
sanity_df = pd.DataFrame({
    "Matrix": ["X_linguistic", "X_embed_minilm", "X_combined"],
    "Shape": [str(X_linguistic.shape), str(X_embed_minilm.shape), str(X_combined.shape)],
    "F1 Mean": [results[k][0] for k in ["X_linguistic", "X_embed_minilm", "X_combined"]],
    "F1 Std": [results[k][1] for k in ["X_linguistic", "X_embed_minilm", "X_combined"]],
})

print("\n=== Sanity Check Summary ===")
sanity_df


=== Sanity Check Summary ===


,Matrix,Shape,F1 Mean,F1 Std
0,X_linguistic,"(70000, 10)",0.661835,0.007224
1,X_embed_minilm,"(70000, 50)",0.673552,0.005990
2,X_combined,"(70000, 60)",0.743365,0.004469


---
## SECTION 5 — Feature Importance Analysis

### 5A. Random Forest Feature Importance (Linguistic Features)

In [34]:
# Train Random Forest on linguistic features
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_linguistic, y)

# Get feature importances
importances = rf.feature_importances_
importance_df = pd.DataFrame({
    "feature": LINGUISTIC_FEATURES,
    "importance": importances
}).sort_values("importance", ascending=False)

print("=== Feature Importances (Random Forest) ===")
importance_df

=== Feature Importances (Random Forest) ===


,feature,importance
5,special_char_ratio,0.160796
4,uppercase_ratio,0.149260
3,sentence_count,0.144629
2,avg_word_length,0.141806
0,char_count,0.137338
1,word_count,0.137315
8,imperative_verb_ratio,0.046715
7,question_mark_count,0.043737
6,injection_keyword_count,0.021039
9,language_switch_flag,0.017364


In [ ]:
# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importances (Linguistic Features)")
plt.gca().invert_yaxis()  # Most important at top
plt.tight_layout()
plt.show()

print(f"\nTop 5 Features:")
for i, row in importance_df.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

### Interpretación de Feature Importances

Los resultados de importancia de características del Random Forest revelan qué features lingüísticas son más discriminativas para distinguir entre prompts benignos y maliciosos:

1. **`injection_keyword_count`** (si aparece alto): Directamente captura la presencia de frases de inyección conocidas. Es la feature más semánticamente relevante.

2. **`char_count` / `word_count`**: Refleja que los contextos con instrucciones embebidas tienden a ser más largos, como se observó en el EDA.

3. **`special_char_ratio`**: Importante dado que el dataset contiene mucho código, tablas markdown y emails, donde los caracteres especiales son delimitadores clave.

4. **`imperative_verb_ratio`**: Captura el tono imperativo de las instrucciones de inyección ("do not", "must", "always").

5. **`uppercase_ratio`**: Puede indicar énfasis en comandos, aunque su importancia puede variar.

**Nota de precaución:** Dado el confounding entre `source` y `label` identificado en el EDA, algunas de estas importancias podrían reflejar diferencias de estilo entre BIPIA y GPT-4o-mini, no necesariamente características universales de inyección.

### 5B. Embedding Visualization with UMAP